# Text-to-SQL Demo

Ask a question in plain English about a database, and get back a SQL query and its result.
This notebook runs the whole text-to-SQL pipeline in-process, with no CLI and no shell scripts.

**Pipeline**
1. **Preprocess**: load the database schema and find database values that match the question.
2. **Schema item classifier** (RoBERTa-large): score every table and column for relevance.
3. **Build the T5 input**: keep the top-k tables and columns, then add matched values and foreign keys.
4. **T5 generator**: beam search produces a SQL skeleton followed by the SQL. The first candidate that executes on the database is returned.

Run the cells from top to bottom. The models load once in section 2, and after that every question is fast.

## 1. Setup

In [1]:
import os, sys, json, re, sqlite3, tempfile, time
from pathlib import Path
from types import SimpleNamespace

# make sure we run from the project root (the folder that contains preprocessing.py)
ROOT = Path.cwd()
if not (ROOT / "preprocessing.py").exists() and (ROOT.parent / "preprocessing.py").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

import numpy as np
import torch
from transformers import (RobertaTokenizerFast, T5TokenizerFast, T5ForConditionalGeneration)
from transformers.trainer_utils import set_seed
from tokenizers import AddedToken

# project modules
from preprocessing import get_db_schemas, get_db_contents
from utils.classifier_model import MyClassifier
from utils.load_dataset import ColumnAndTableClassifierDataset
from schema_item_classifier import prepare_batch_inputs_and_labels
from text2sql_data_generator import prepare_input_and_output
from utils.text2sql_decoding_utils import decode_sqls
from torch.utils.data import DataLoader

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: c:\Users\bkska\Documents\nlp_ppt\project
torch 1.11.0+cu113 | CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [2]:
# ---------------- configuration ----------------
CLASSIFIER_PATH = "./models/text2sql_schema_item_classifier"
T5_PATH         = "./models/text2sql-t5-base/checkpoint-39312"
TABLES_JSON     = "./data/spider/tables.json"
DEV_JSON        = "./data/spider/dev.json"
DB_PATH         = "./database"

TOPK_TABLES  = 4     # top-k tables kept
TOPK_COLUMNS = 5     # top-k columns kept per table
NUM_BEAMS    = 8     # lower (e.g. 4) if you run out of GPU memory
SEED         = 42

for p in [CLASSIFIER_PATH, T5_PATH, TABLES_JSON, DEV_JSON, DB_PATH]:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)

OK      ./models/text2sql_schema_item_classifier
OK      ./models/text2sql-t5-base/checkpoint-39312
OK      ./data/spider/tables.json
OK      ./data/spider/dev.json
OK      ./database


## 2. Load the models (once)
This takes a minute or two the first time.

In [3]:
set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# stage 1: schema item classifier
cls_tokenizer = RobertaTokenizerFast.from_pretrained(CLASSIFIER_PATH, add_prefix_space=True)
cls_model = MyClassifier(model_name_or_path=CLASSIFIER_PATH, vocab_size=len(cls_tokenizer), mode="test")
cls_model.load_state_dict(torch.load(CLASSIFIER_PATH + "/dense_classifier.pt", map_location="cpu"))
cls_model = cls_model.to(device).eval()

# stage 2: T5 text-to-SQL generator
t5_tokenizer = T5TokenizerFast.from_pretrained(T5_PATH, add_prefix_space=True)
t5_tokenizer.add_tokens([AddedToken(" <="), AddedToken(" <")])
t5_model = T5ForConditionalGeneration.from_pretrained(T5_PATH).to(device).eval()

# database schemas
db_schemas = get_db_schemas(json.load(open(TABLES_JSON)))
print("Models loaded. Databases available:", len(db_schemas))

Models loaded. Databases available: 166


## 3. The pipeline
Each function is one stage of the pipeline.

In [4]:
def db_file(db_id):
    return f"{DB_PATH}/{db_id}/{db_id}.sqlite"


def preprocess(question, db_id):
    """Stage 0: schema + matched database values (what preprocessing.py does in test mode)."""
    question = (question.replace("\u2018", "'").replace("\u2019", "'")
                        .replace("\u201c", "'").replace("\u201d", "'").strip())
    schema = db_schemas[db_id]
    data = {"question": question, "db_id": db_id,
            "sql": "", "norm_sql": "", "sql_skeleton": "",
            "db_schema": [], "pk": schema["pk"], "fk": schema["fk"],
            "table_labels": [], "column_labels": []}
    for table in schema["schema_items"]:
        contents = get_db_contents(question, table["table_name_original"],
                                   table["column_names_original"], db_id, DB_PATH)
        data["db_schema"].append({
            "table_name_original": table["table_name_original"],
            "table_name": table["table_name"],
            "column_names": table["column_names"],
            "column_names_original": table["column_names_original"],
            "column_types": table["column_types"],
            "db_contents": contents})
        data["table_labels"].append(0)                                   # no labels at test time
        data["column_labels"].append([0] * len(table["column_names_original"]))
    return data


@torch.no_grad()
def rank_schema(data):
    """Stage 1: probability that each table / column is needed for the question."""
    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "one.json")
        with open(path, "w", encoding="utf-8") as f:
            json.dump([data], f, ensure_ascii=False)
        ds = ColumnAndTableClassifierDataset(dir_=path, use_contents=True, add_fk_info=True)
    batch = [ds[0]]
    (input_ids, attn, col_labels, tab_labels, q_ids, col_ids, tab_ids,
     col_counts) = prepare_batch_inputs_and_labels(batch, cls_tokenizer)
    out = cls_model(input_ids.to(device), attn.to(device), q_ids, col_ids, tab_ids, col_counts)

    table_probs = torch.softmax(out["batch_table_name_cls_logits"][0], dim=1)[:, 1].cpu().tolist()
    col_probs_flat = torch.softmax(out["batch_column_info_cls_logits"][0], dim=1)[:, 1].cpu().tolist()
    counts = col_counts[0]
    column_probs = [col_probs_flat[sum(counts[:i]):sum(counts[:i + 1])] for i in range(len(counts))]

    # very large schemas get truncated at 512 tokens; unseen items get -1 (ranked last)
    n_tab = len(data["db_schema"])
    table_probs += [-1] * (n_tab - len(table_probs))
    full = []
    for i in range(n_tab):
        n_col = len(data["db_schema"][i]["column_names_original"])
        p = column_probs[i] if i < len(column_probs) else []
        full.append(p + [-1] * (n_col - len(p)))
    return table_probs, full


def build_t5_input(data, table_probs, column_probs):
    """Stage 2: keep top-k tables / columns and serialize question + schema (text2sql_data_generator.py)."""
    opt = SimpleNamespace(use_contents=True, add_fk_info=True, output_skeleton=True)
    tp = [round(x, 4) for x in table_probs]
    top_tables = np.argsort(-np.array(tp), kind="stable")[:TOPK_TABLES].tolist()

    ranked = {"question": data["question"], "db_id": data["db_id"], "db_schema": [],
              "sql_skeleton": "", "norm_sql": ""}
    kept = []
    for t in top_tables:
        cp = [round(x, 2) for x in column_probs[t]]
        top_cols = np.argsort(-np.array(cp), kind="stable")[:TOPK_COLUMNS].tolist()
        tbl = data["db_schema"][t]
        ranked["db_schema"].append({
            "table_name_original": tbl["table_name_original"],
            "column_names_original": [tbl["column_names_original"][c] for c in top_cols],
            "db_contents": [tbl["db_contents"][c] for c in top_cols]})
        kept.append((tbl["table_name_original"], round(table_probs[t], 3),
                     [(tbl["column_names_original"][c], round(column_probs[t][c], 3)) for c in top_cols]))

    names = [t["table_name_original"] for t in data["db_schema"]]
    ranked["fk"] = [fk for fk in data["fk"]
                    if names.index(fk["source_table_name_original"]) in top_tables
                    and names.index(fk["target_table_name_original"]) in top_tables]
    input_sequence, _ = prepare_input_and_output(opt, ranked)
    tc_original = [t["table_name_original"] + "." + c
                   for t in ranked["db_schema"] for c in t["column_names_original"] + ["*"]]
    return input_sequence, tc_original, kept


@torch.no_grad()
def generate_sql(input_sequence, tc_original, db_id):
    """Stage 3: beam search with T5; return the first candidate that executes on the database."""
    enc = t5_tokenizer([input_sequence], return_tensors="pt", padding="max_length",
                       max_length=512, truncation=True).to(device)
    out = t5_model.generate(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
                            max_length=256, decoder_start_token_id=t5_model.config.decoder_start_token_id,
                            num_beams=NUM_BEAMS, num_return_sequences=NUM_BEAMS)
    out = out.view(1, NUM_BEAMS, out.shape[1])
    sql = decode_sqls(DB_PATH, out, [db_id], [input_sequence], t5_tokenizer, [tc_original])[0]
    top_beam = t5_tokenizer.decode(out[0, 0, :], skip_special_tokens=True)
    skeleton = top_beam.split("|")[0].strip() if "|" in top_beam else ""
    return sql, skeleton


def run_sql(sql, db_id, limit=None):
    con = sqlite3.connect(db_file(db_id))
    con.text_factory = lambda b: b.decode(errors="ignore")
    try:
        cur = con.execute(sql)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = cur.fetchmany(limit) if limit else cur.fetchall()
        return cols, rows
    finally:
        con.close()


def text_to_sql(question, db_id, verbose=True, show_rows=10):
    """Full pipeline: question + database id -> SQL (+ result)."""
    if db_id not in db_schemas:
        raise ValueError(f"Unknown db_id '{db_id}'. Use list_databases() to see valid ids.")
    t0 = time.time()
    data = preprocess(question, db_id)
    table_probs, column_probs = rank_schema(data)
    input_sequence, tc_original, kept = build_t5_input(data, table_probs, column_probs)
    sql, skeleton = generate_sql(input_sequence, tc_original, db_id)
    elapsed = time.time() - t0

    result = {"question": question, "db_id": db_id, "sql": sql, "skeleton": skeleton,
              "kept_schema": kept, "t5_input": input_sequence, "seconds": elapsed}
    if verbose:
        print("Question :", question)
        print("Database :", db_id)
        print("\nSchema items kept by the classifier (probability):")
        for tname, tp, cols in kept:
            print(f"  {tname} ({tp})")
            print("     " + ", ".join(f"{c} ({p})" for c, p in cols))
        print("\nSQL skeleton:", skeleton)
        print("Predicted SQL:\n  " + sql)
        try:
            cols, rows = run_sql(sql, db_id, show_rows)
            print(f"\nResult ({len(cols)} column(s)): {cols}")
            for r in rows:
                print("  ", r)
        except Exception as e:
            print("\n(could not execute the SQL:", e, ")")
        print(f"\n[{elapsed:.1f}s]")
    return result


def list_databases():
    return sorted(db_schemas)


def show_schema(db_id):
    for t in db_schemas[db_id]["schema_items"]:
        print(f"{t['table_name_original']}: " + ", ".join(t["column_names_original"]))

## 4. Explore the databases

In [5]:
dbs = list_databases()
print(len(dbs), "databases, e.g.:", dbs[:15])
show_schema("concert_singer")

166 databases, e.g.: ['academic', 'activity_1', 'aircraft', 'allergy_1', 'apartment_rentals', 'architecture', 'assets_maintenance', 'baseball_1', 'battle_death', 'behavior_monitoring', 'bike_1', 'body_builder', 'book_2', 'browser_web', 'candidate_poll']
stadium: stadium_id, location, name, capacity, highest, lowest, average
singer: singer_id, name, country, song_name, song_release_year, age, is_male
concert: concert_id, concert_name, theme, stadium_id, year
singer_in_concert: concert_id, singer_id


## 5. Ask a question
Change `question` and `db_id` and re-run the cell.

In [6]:
question = "How many singers do we have?"
db_id = "concert_singer"
res = text_to_sql(question, db_id)

Question : How many singers do we have?
Database : concert_singer

Schema items kept by the classifier (probability):
  singer (0.973)
     singer_id (0.02), name (0.009), country (0.009), age (0.009), song_name (0.002)
  stadium (0.018)
     location (0.009), name (0.009), capacity (0.009), highest (0.009), lowest (0.009)
  concert (0.018)
     theme (0.009), year (0.009), concert_id (0.001), concert_name (0.002), stadium_id (0.001)
  singer_in_concert (0.018)
     concert_id (0.001), singer_id (0.001)

SQL skeleton: select count ( _ ) from _
Predicted SQL:
  select count ( * ) from singer

Result (1 column(s)): ['count ( * )']
   (6,)

[6.4s]


In [7]:
res = text_to_sql("What are the names of singers who are older than 40?", "concert_singer")

Question : What are the names of singers who are older than 40?
Database : concert_singer

Schema items kept by the classifier (probability):
  singer (0.981)
     name (0.989), age (0.989), country (0.009), singer_id (0.002), song_name (0.002)
  stadium (0.018)
     location (0.009), name (0.009), capacity (0.009), highest (0.009), lowest (0.009)
  concert (0.018)
     theme (0.009), year (0.009), concert_id (0.001), concert_name (0.002), stadium_id (0.001)
  singer_in_concert (0.018)
     concert_id (0.001), singer_id (0.001)

SQL skeleton: select _ from _ where _
Predicted SQL:
  select name from singer where age > 40

Result (1 column(s)): ['Name']
   ('Joe Sharp',)
   ('Rose White',)
   ('John Nizinik',)

[2.4s]


In [8]:
res = text_to_sql("Show the name and the release year of the song by the youngest singer.", "concert_singer")

Question : Show the name and the release year of the song by the youngest singer.
Database : concert_singer

Schema items kept by the classifier (probability):
  singer (0.981)
     song_release_year (0.999), song_name (0.994), age (0.988), name (0.165), country (0.009)
  stadium (0.018)
     location (0.009), name (0.009), capacity (0.009), highest (0.009), lowest (0.009)
  concert (0.018)
     theme (0.009), year (0.009), concert_id (0.001), concert_name (0.002), stadium_id (0.001)
  singer_in_concert (0.018)
     concert_id (0.001), singer_id (0.001)

SQL skeleton: select _ from _ order by _ asc limit _
Predicted SQL:
  select song_name , song_release_year from singer order by age asc limit 1

Result (2 column(s)): ['Song_Name', 'Song_release_year']
   ('Love', '2016')

[5.2s]


## 6. Try your own

In [9]:
MY_DB = "concert_singer"          # pick from list_databases()
MY_QUESTION = "List the stadium names and their capacities, largest first."
show_schema(MY_DB)
print()
res = text_to_sql(MY_QUESTION, MY_DB)

stadium: stadium_id, location, name, capacity, highest, lowest, average
singer: singer_id, name, country, song_name, song_release_year, age, is_male
concert: concert_id, concert_name, theme, stadium_id, year
singer_in_concert: concert_id, singer_id

Question : List the stadium names and their capacities, largest first.
Database : concert_singer

Schema items kept by the classifier (probability):
  stadium (0.981)
     capacity (0.987), name (0.985), location (0.009), highest (0.01), lowest (0.009)
  singer (0.018)
     name (0.009), country (0.009), age (0.009), singer_id (0.001), song_name (0.002)
  concert (0.018)
     theme (0.009), year (0.009), concert_id (0.001), concert_name (0.002), stadium_id (0.001)
  singer_in_concert (0.018)
     concert_id (0.001), singer_id (0.001)

SQL skeleton: select _ from _ order by _ desc limit _
Predicted SQL:
  select name , capacity from stadium order by capacity desc limit 1

Result (2 column(s)): ['Name', 'Capacity']
   ('Hampden Park', 52500)


## 7. Quick accuracy check on Spider dev
Runs the pipeline on the first `N` dev questions and compares the **execution result** of the prediction
with the result of the gold SQL. This is a quick approximation of Spider's execution accuracy, and it
doesn't need the `third_party` scoring repos. Keep `N` small on a laptop GPU.

In [10]:
N = 20
dev = json.load(open(DEV_JSON))[:N]

def same_result(pred_sql, gold_sql, db_id):
    try:
        _, a = run_sql(pred_sql, db_id)
        _, b = run_sql(gold_sql, db_id)
    except Exception:
        return False
    return sorted(map(repr, a)) == sorted(map(repr, b))

correct = 0
for i, ex in enumerate(dev, 1):
    r = text_to_sql(ex["question"], ex["db_id"], verbose=False)
    ok = same_result(r["sql"], ex["query"], ex["db_id"])
    correct += ok
    print(f"[{i:02d}] {'OK ' if ok else 'BAD'} {ex['question']}")
    if not ok:
        print("       pred:", r["sql"])
        print("       gold:", ex["query"])
print(f"\nExecution match: {correct}/{len(dev)} = {correct/len(dev):.1%}")

[01] OK  How many singers do we have?
[02] OK  What is the total number of singers?
[03] OK  Show name, country, age for all singers ordered by age from the oldest to the youngest.
[04] OK  What are the names, countries, and ages for every singer in descending order of age?
[05] OK  What is the average, minimum, and maximum age of all singers from France?
[06] OK  What is the average, minimum, and maximum age for all French singers?
[07] OK  Show the name and the release year of the song by the youngest singer.
[08] OK  What are the names and release years for all the songs of the youngest singer?
[09] OK  What are all distinct countries where singers above age 20 are from?
[10] OK  What are  the different countries with singers above age 20?
[11] OK  Show all countries and the number of singers in each country.
[12] OK  How many singers are from each country?
[13] OK  List all song names by singers above the average age.
[14] OK  What are all the song names by singers who are older th